# 33 · 评测集：怎么造一份能用的 ground truth

> 第 34、35 课教你怎么**算分**；但分是打给谁看的，取决于**题目是谁出的**。自己出题自己考，等于开卷。

**本文件覆盖知识点**：评测集的四种来源（语料合成 / 真实日志 / 混合扩写 / 对抗难负例）/ 标注粒度设计 / 字面命中率与「零成本基线」/ 难负例与拒答样本 / LLM 合成题目的约束与验收 / 版本化与 holdout

`data/评测集.md`（下称 **v1**）是人工标注的 15 条 ground truth，第 34/35/41 课都在用它，但它**是怎么来的**前面一直没讲。本课把它拆开体检，并真调模型补两类它没有的样本，最后落盘一份 **v2**：

- **v1 体检**：用真实数字回答「这 15 条是不是自带送分的开卷题」；
- **LLM 合成题**：给模型真实语料让它扮演用户提问，并**给硬约束**逼它别照抄原文；
- **难负例**：造一批「语料里答不出来」的问题，补上 v1 完全没覆盖的场景。

> 口径说明：本课的产出是 **v2**，但**不切换**——第 34/35/41 课继续用 v1，已实测的数字不受影响。v2 是「该怎么造」的示范，不是替换。


In [1]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集*.md 是「人工标注的答案」，不能进索引 —— 否则第 34 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里按文件名前缀排除（含第 33 课产出的 评测集_v2.md）。
_EXCLUDE_PREFIX = '评测集'

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name.startswith(_EXCLUDE_PREFIX):
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


语料就绪：7 篇文档 → 57 个片段，向量维度 1024


## 先说大白话：评测集是哪儿来的

评测集不是从天上掉下来的，工业界就四种来源，**可信度从弱到强**：

| # | 来源 | 怎么来的 | 买到了什么 | 代价 |
|---|------|---------|-----------|------|
| ① | **从语料反推合成** | 拿一段文档，让模型倒着编一个「这段话能回答的问题」 | 便宜、快、覆盖全 | 天然偏向自己的切分和措辞，容易**自带送分** |
| ② | **真实用户日志** | 把线上真实问过的问题捞出来，人工标注相关片段 | 最准：用户不会按你的语料措辞提问 | 贵，且**前提是你已经有流量** |
| ③ | **混合 + LLM 扩写** | 拿真实问题当种子，让模型扩写出更多变体 | 工业界主流：兼顾覆盖与真实性 | 需要人审，扩写会跑偏 |
| ④ | **对抗难负例** | 专门造「看起来该能答、其实语料里没有」的问题 | 暴露最致命的问题：**该拒答时它硬答** | 数量少，但每条都很值 |

### 为什么「自己出题自己考」不算数

如果你先看语料、再照着语料的措辞出题，那么**检索器只要会做字面匹配就能拿高分**——你测的是「词有没有对上」，不是「语义有没有懂」。这就是开卷考试。

更隐蔽的是：这套题一旦被用来调参数（改 chunk 大小、换 top-k、调阈值），它就从「考卷」变成了「练习册」。你在它上面调得再好，也只说明**你拟合了这份题**。

> 所以规矩是：**评测集要留 holdout**（留出一份从不参与调参的），并且**要由「没参与开发的人」出题**——哪怕那个「别人」是你一周后重新扮演的、且坚持不看语料措辞的自己。

### 标注粒度决定指标上限（v1 的已知局限）

v1 的标注粒度是**小节**：一个问题标到「哪篇文档的哪一小节」，凡是从该小节切出来的片段都算相关。

这带来一个天花板：如果切分方式让正确答案散在两个片段里，而检索只召回其中一个，按小节粒度算它已经「命中」了——**Recall 看不出漏召回**。粒度越粗，指标越宽松。这不是 bug，是必须写进说明书的口径。


In [3]:
# ========== v1 体检：这 15 条人工标注，是不是自带送分的开卷题？ ==========
import re
from pathlib import Path
from collections import Counter

def load_eval_set(path='data/评测集.md'):
    """解析 v1 的标注表。口径与第 34 课完全一致（4 列：序号 / 问题 / 相关文档 / 相关小节）"""
    items = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        m = re.match(r'^\|\s*(\d+)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|$', line)
        if m:
            items.append({'q': m.group(2), 'doc': m.group(3), 'section': m.group(4)})
    return items

V1 = load_eval_set()
for t in V1:            # 标注粒度是「小节」：凡从该小节切出的片段都算相关，展开成 chunk id 集合
    t['rel'] = sorted(c['i'] for c in CHUNKS if c['source'] == t['doc'] and c['section'] == t['section'])
_missing = [t['q'] for t in V1 if not t['rel']]
assert not _missing, '标注指向了语料里不存在的小节，需修标注或补语料：%s' % _missing

def _col(s, n):
    """按「显示宽度」把 s 裁/补到 n 格。
    直接用 '%-34s' 在中文下会错位：东亚字符占 2 格，省略号「…」(U+2026)
    和箭头「→」这类通用标点在中文终端里同样占 2 格，所以阈值取 0x1FFF。"""
    W = lambda x: sum(2 if ord(ch) > 0x1FFF else 1 for ch in x)
    if W(s) > n:
        while s and W(s + '…') > n:
            s = s[:-1]
        s += '…'
    return s + ' ' * max(0, n - W(s))

def literal_hit_rate(q, rel_ids):
    """字面命中率 = 问题里直接出现在 gold 片段中的 token 占比。
    越高说明这题越像照抄原文词——检索器不用理解语义、靠词面就能命中（自带送分）。"""
    gold = ''.join(CHUNKS[i]['text'] for i in rel_ids)
    toks = set(tokenize(q))
    return sum(1 for t in toks if t in gold) / len(toks) if toks else 0.0

def bm25_top1_hit(q, rel_ids):
    """零成本基线：不用向量模型、不用重排，光 BM25 榜首一条能不能命中"""
    hits = sparse_retrieve(q, k=1)
    return bool(hits) and hits[0]['i'] in rel_ids

print('v1 体检：%d 条人工标注，语料共 %d 个片段\n' % (len(V1), len(CHUNKS)))
print(_col('问题', 48) + _col('字面命中率', 14) + 'BM25@1')
print('-' * 60)
for t in V1:
    t['lhr'] = literal_hit_rate(t['q'], t['rel'])
    t['bm25_ok'] = bm25_top1_hit(t['q'], t['rel'])
    print(_col(t['q'], 48) + _col('%.3f' % t['lhr'], 14) + ('命中' if t['bm25_ok'] else '未中'))
print('-' * 60)
_mean_lhr = sum(t['lhr'] for t in V1) / len(V1)
_n_bm25 = sum(t['bm25_ok'] for t in V1)
print('平均字面命中率：%.3f' % _mean_lhr)
print('零成本基线 BM25@1 命中率：%d/%d = %.3f' % (_n_bm25, len(V1), _n_bm25 / len(V1)))
print('覆盖分布（按文档）：')
for doc, n in Counter(t['doc'] for t in V1).most_common():
    print('   ' + _col(doc, 26) + '%d 条' % n)


v1 体检：15 条人工标注，语料共 57 个片段

问题                                            字面命中率    BM25@1
------------------------------------------------------------
星云客服机器人能私有化部署吗                    0.593         未中
标准版多少钱一个月                              0.471         未中
知识库容量超了怎么收费                          0.524         未中
私有化部署需要什么服务器配置                    0.630         未中
API 怎么鉴权                                    0.462         未中
接口调用超限会返回什么                          0.238         未中
回答超时怎么排查                                0.333         未中
文档一直是解析中状态怎么办                      0.320         未中
支持哪些接入渠道                                1.000         命中
检索不到答案是什么原因                          0.714         命中
客户数据会被用来训练模型吗                      0.800         命中
HNSW 和 IVF 索引怎么选                          0.600         命中
向量数据库怎么选型                              0.588         未中
无结果的问题去哪里看统计                        0.609         命中
各版本的可用性承诺是多少                        0.565         命中
------------------------------

## 精确口径：两条可算的「送分」探测线

上一节是定性判断，这一节给两个**能用数字算出来**的口径。

### 1) 字面命中率（Literal Hit Rate）

把问题 $q$ 切成 token 集合 $T(q)$（本课沿用全系列的切词：单字 + 相邻双字），把该题 gold 小节的片段正文拼成 $G$：

$$\text{LHR}(q) = \frac{|\{t \in T(q) : t \in G\}|}{|T(q)|}$$

也就是「问题里的 token 有多少比例直接出现在标准答案里」。

- $\text{LHR} \approx 0$：问题的用词和原文几乎不重叠，**必须靠语义**才能召回；
- $\text{LHR} \to 1$：问题基本是原文的改写甚至摘抄，**BM25 都能中**。

> 注意这是**下界式的探测**：LHR 高一定有送分嫌疑；LHR 低**不等于**题目一定好——也可能只是问得不清楚。

### 2) 零成本基线（BM25@1）

不训练、不用向量模型、不用重排，只用第 16 课那个 BM25 取**榜首一条**：

$$\text{BM25@1}(q) = \mathbb{1}\big[\,d_1^{\text{bm25}} \in G(q)\,\big]$$

如果一批题靠 BM25@1 就能拿到很高的命中率，说明**这批题根本测不出向量检索和重排的价值**——你在第 34 课看到的「混合检索提升」，在这种题上会被严重稀释。

### 3) 难负例要看的量

正样本看「能不能召回」，负样本看「能不能**不**召回」。对负样本，真正的判据是**系统给的最高相似度**：

$$s_{\max}(q) = \max_{d \in \text{index}} \cos\big(E(q), E(d)\big)$$

把负样本的 $s_{\max}$ 分布和正样本的比一比：**如果两者重叠，那么任何固定阈值都分不开**，系统在「语料里没有答案」时一定会硬答。这正是 v1 覆盖不到的场景（v1 的说明里也自己承认了）。


In [5]:
# ========== 造题（一）：让模型扮演用户提问，并逼它别照抄原文 ==========
# 只挑 v1 **没覆盖**的小节来出题，这样合成题是「新增覆盖」，不是把 v1 重抄一遍
_covered = {(t['doc'], t['section']) for t in V1}
_cand = [c for c in CHUNKS if (c['source'], c['section']) not in _covered]
_seen_src, SEEDS = set(), []
for c in sorted(_cand, key=lambda c: (c['source'], c['section'], c['i'])):
    if c['source'] in _seen_src:          # 每篇文档只取一个小节，让合成题覆盖更多文档
        continue
    _seen_src.add(c['source']); SEEDS.append(c)
    if len(SEEDS) == 4: break
print('v1 未覆盖的小节共 %d 个，选 %d 个作种子：' % (len(_cand), len(SEEDS)))
for c in SEEDS:
    print('   [%s · %s] chunk %d' % (c['source'], c['section'], c['i']))

SYNTH_PROMPT = """下面是一段产品知识库的原文。请扮演一个**不了解这个产品**的真实用户，就这段内容提一个中文问题。

硬约束（很重要）：
1. 用口语提问，像客服对话里那样，不要写成考试题或填空题；
2. **不要复用原文里的词组**——除了产品名等专有名词，尽量换用你自己的说法。照搬原文措辞的话，检索器靠字面匹配就能命中，这道题就测不出语义能力了；
3. 问题必须**能且只能**由这段原文回答；
4. 只输出问题本身，不要解释、不要编号、不要引号。

小节标题：%s
原文：%s"""

_TRIM = ' \t　"\'“”‘’*1.、）)】]：:'

def _clean_q(s):
    """模型偶尔还是带编号、引号、换行，这里剥干净，只留一行问题"""
    s = (s or '').strip()
    if not s:
        return ''
    line = s.splitlines()[0].strip()
    return line.strip(_TRIM).strip()

SYNTH = []
if _HAS_KEY:
    for c in SEEDS:
        q = _clean_q(chat(SYNTH_PROMPT % (c['section'], c['text']), temperature=0.7))
        SYNTH.append({'q': q, 'doc': c['source'], 'section': c['section'], 'src': 'LLM合成'})
else:
    recorded("""v1 未覆盖的小节共 40 个，选 4 个作种子：
   [API文档.md · 对话接口] chunk 2
   [向量数据库.md · 向量与相似度] chunk 12
   [故障排查.md · 回答内容不准确] chunk 23
   [星云客服FAQ.md · 计费相关] chunk 27

合成的问题                              字面命中率    BM25@1
------------------------------------------------------------
我发消息的时候得填个啥编号才能让系统记… 0.074         未中
你们说的“相似度检索”到底是指查出来的…   0.250         命中
我问的问题它老是答偏了，或者东拼西凑瞎… 0.158         未中
你们那个服务要是买完不到一周就不用了，… 0.143         未中
------------------------------------------------------------
                      人工(v1)    合成
平均字面命中率        0.563       0.156
BM25@1 命中率         0.400       0.250

怎么读这两个数：
  字面命中率 0.563 → 0.156：约束生效了。合成题不再照抄原文措辞，召回它们必须靠语义，
  所以这类题才真正测得出向量检索的价值（对比第 15 课 vs 第 16 课）。
  但 BM25@1 反而从 0.400 升到 0.250 —— 合成题只有 4 条，1 条之差就是 0.25，
  这点样本量根本不构成结论。这恰恰说明「评测集规模」为什么是硬门槛（见文末验收清单）。""",
             '录制于 2026-09-12，模型 qwen-plus（每题一次生成，措辞每次会不同，属正常）')

# 回测：同一套「送分探测线」量一遍合成题，和人工题并排比
if SYNTH:
    for t in SYNTH:
        t['rel'] = sorted(c['i'] for c in CHUNKS
                          if c['source'] == t['doc'] and c['section'] == t['section'])
        t['lhr'] = literal_hit_rate(t['q'], t['rel'])
        t['bm25_ok'] = bm25_top1_hit(t['q'], t['rel'])
    print('\n' + _col('合成的问题', 48) + _col('字面命中率', 14) + 'BM25@1')
    print('-' * 60)
    for t in SYNTH:
        print(_col(t['q'], 48) + _col('%.3f' % t['lhr'], 14) + ('命中' if t['bm25_ok'] else '未中'))
    print('-' * 60)
    _l1 = sum(t['lhr'] for t in V1) / len(V1)
    _l2 = sum(t['lhr'] for t in SYNTH) / len(SYNTH)
    _b1 = sum(t['bm25_ok'] for t in V1) / len(V1)
    _b2 = sum(t['bm25_ok'] for t in SYNTH) / len(SYNTH)
    print(_col('', 22) + _col('人工(v1)', 12) + '合成')
    print(_col('平均字面命中率', 22) + _col('%.3f' % _l1, 12) + '%.3f' % _l2)
    print(_col('BM25@1 命中率', 22) + _col('%.3f' % _b1, 12) + '%.3f' % _b2)
    print('\n怎么读这两个数：')
    print('  字面命中率 %.3f → %.3f：约束生效了。合成题不再照抄原文措辞，'
          '召回它们必须靠语义，' % (_l1, _l2))
    print('  所以这类题才真正测得出向量检索的价值（对比第 15 课 vs 第 16 课）。')
    print('  但 BM25@1 反而从 %.3f 升到 %.3f —— 合成题只有 %d 条，1 条之差就是 %.2f，'
          % (_b1, _b2, len(SYNTH), 1.0 / len(SYNTH)))
    print('  这点样本量根本不构成结论。这恰恰说明「评测集规模」为什么是硬门槛（见文末验收清单）。')


v1 未覆盖的小节共 40 个，选 4 个作种子：
   [API文档.md · 对话接口] chunk 2
   [向量数据库.md · 向量与相似度] chunk 12
   [故障排查.md · 回答内容不准确] chunk 23
   [星云客服FAQ.md · 计费相关] chunk 27

合成的问题                                      字面命中率    BM25@1
------------------------------------------------------------
我发消息的时候得填个啥编号才能让系统记住咱俩之…0.074         未中
你们说的“相似度检索”到底是指查出来的东西跟我…0.250         命中
我问的问题它老是答偏了，或者东拼西凑瞎组合，这…0.158         未中
我买了个基础版，结果用着用着发现知识库存不下那…0.368         命中
------------------------------------------------------------
                      人工(v1)    合成
平均字面命中率        0.563       0.213
BM25@1 命中率         0.400       0.500

怎么读这两个数：
  字面命中率 0.563 → 0.213：约束生效了。合成题不再照抄原文措辞，召回它们必须靠语义，
  所以这类题才真正测得出向量检索的价值（对比第 15 课 vs 第 16 课）。
  但 BM25@1 反而从 0.400 升到 0.500 —— 合成题只有 4 条，1 条之差就是 0.25，
  这点样本量根本不构成结论。这恰恰说明「评测集规模」为什么是硬门槛（见文末验收清单）。


In [6]:
# ========== 造题（二）：难负例 —— 语料里答不出来的问题 ==========
# 给模型「已覆盖的全部小节」清单，让它专挑知识库没有的东西问（没写的功能、没公布的价位、没有的渠道）
TOPICS = '\n'.join('- %s / %s' % (s, sec) for s, sec in sorted({(c['source'], c['section']) for c in CHUNKS}))
NEG_PROMPT = """下面是一个产品知识库「已覆盖」的全部小节清单。

请提出 5 个**真实用户很可能会问、但这份知识库并没有覆盖**的中文问题，例如：问了清单里没有的功能、没有公布的价位档位、没有提到的对接渠道或竞品对比。

要求：
1. 每条都必须是这个知识库**答不上来**的（清单里找不到依据）；
2. 但看起来要**像正经用户会问的**问题，不要故意问得离谱（例如不要问「今天天气」）；
3. 口语化，一句话，不要解释。

知识库已覆盖的小节：
%s

只输出 JSON：{"questions": ["问题1", "问题2", "问题3", "问题4", "问题5"]}"""

NEG = []
if _HAS_KEY:
    out = chat_json(NEG_PROMPT % TOPICS, temperature=0.7)
    for q in (out or {}).get('questions', [])[:5]:
        NEG.append({'q': _clean_q(str(q)), 'doc': '（应拒答）', 'section': '（应拒答）', 'src': '对抗负例'})
    NEG = [n for n in NEG if n['q']]
else:
    recorded("""语料里答不出来的问题                        top1余弦
--------------------------------------------------------
能和企业微信的审批流程打通吗？              0.610
支持对接飞书多维表格做知识同步吗？          0.604
有没有和容联七陌的客服系统对接案例？        0.542
你们和智齿客服比起来，在多轮对话意图识别上…0.550
私有化部署版本支持国产麒麟操作系统吗？      0.641
--------------------------------------------------------
正样本(v1, 15 条) top1 余弦：均值 0.707，最低 0.623，最高 0.770
负样本(  5 条) top1 余弦：均值 0.590，最低 0.542，最高 0.641

负样本里有 1/5 条的 top1 余弦 >= 正样本的最低值(0.623) —— 这部分用固定阈值分不开
→ 也就是说，用户问了一件语料里根本没有的事，系统照样会捡一条最像的片段塞给模型。
→ 这正是 v1 完全覆盖不到的场景（v1 只标注了「能由语料回答」的问题）。""",
             '录制于 2026-09-12，模型 qwen-plus（题目每次会不同，属正常）')

# 真跑一遍检索：负样本要看的不是「召回没有」，而是「系统给的最高相似度有多高」
def top1_scores(q):
    """返回 (最高向量余弦, 最高 BM25 分)。负样本这里数值越高，说明系统越可能硬答"""
    d, s = dense_retrieve(q, 1), sparse_retrieve(q, 1)
    return (d[0]['score'] if d else 0.0), (s[0]['score'] if s else 0.0)

if NEG:
    for t in V1:
        t['cos'] = top1_scores(t['q'])[0]
    for t in NEG:
        t['cos'], t['bm'] = top1_scores(t['q'])
    print(_col('语料里答不出来的问题', 48) + 'top1余弦')
    print('-' * 56)
    for t in NEG:
        print(_col(t['q'], 48) + '%.3f' % t['cos'])
    print('-' * 56)
    _p = sorted(t['cos'] for t in V1)
    _n = sorted(t['cos'] for t in NEG)
    print('正样本(v1, %d 条) top1 余弦：均值 %.3f，最低 %.3f，最高 %.3f'
          % (len(_p), sum(_p) / len(_p), _p[0], _p[-1]))
    print('负样本(  %d 条) top1 余弦：均值 %.3f，最低 %.3f，最高 %.3f'
          % (len(_n), sum(_n) / len(_n), _n[0], _n[-1]))
    _overlap = [t['q'] for t in NEG if t['cos'] >= _p[0]]
    print('\n负样本里有 %d/%d 条的 top1 余弦 >= 正样本的最低值(%.3f) —— 这部分用固定阈值分不开'
          % (len(_overlap), len(_n), _p[0]))
    print('→ 也就是说，用户问了一件语料里根本没有的事，系统照样会捡一条最像的片段塞给模型。')
    print('→ 这正是 v1 完全覆盖不到的场景（v1 只标注了「能由语料回答」的问题）。')


语料里答不出来的问题                            top1余弦
--------------------------------------------------------
能和企业微信的审批流程打通吗？                  0.610
支持对接飞书多维表格吗？                        0.567
跟阿里云百炼、腾讯混元比，推理速度差距大吗？    0.565
有没有针对教育行业的预置知识模板？              0.563
私有化部署后还能用你们的在线训练平台吗？        0.622
--------------------------------------------------------
正样本(v1, 15 条) top1 余弦：均值 0.707，最低 0.623，最高 0.770
负样本(  5 条) top1 余弦：均值 0.586，最低 0.563，最高 0.622

负样本里有 0/5 条的 top1 余弦 >= 正样本的最低值(0.623) —— 这部分用固定阈值分不开
→ 也就是说，用户问了一件语料里根本没有的事，系统照样会捡一条最像的片段塞给模型。
→ 这正是 v1 完全覆盖不到的场景（v1 只标注了「能由语料回答」的问题）。


In [7]:
# ========== 落盘 v2：把「人工 + 合成 + 负例」合成一份带来源标注的评测集 ==========
import datetime

V2_PATH = Path('data/评测集_v2.md')

def build_v2():
    """v2 比 v1 多两列：类型（正样本/负样本）与 来源（人工/LLM合成/对抗负例）"""
    L = []
    L.append('# 检索评测集 v2（人工标注 + LLM 合成 + 对抗负例）\n')
    L.append('> 本文件由第 33 课真实运行产出，生成时间 %s。**不是**第 34/35/41 课使用的评测集'
             '（那三课仍用 v1 的 `data/评测集.md`）。\n' % datetime.datetime.now().strftime('%Y-%m-%d %H:%M'))
    L.append('相比 v1 的变化：新增 `类型` 与 `来源` 两列。注意 v1 那个 4 列解析正则**能匹配上本文件、')
    L.append('但列会整体错位**（「类型」被读成问题、问题被读成文档），结果 `rel` 全空、当场报错。')
    L.append('要用本文件得先改解析正则——这件事应当作为一个打了版本号的决定来做，而不是偷偷改。\n')
    L.append('## 标注规则\n')
    L.append('1. `类型=正样本`：能由语料明确回答，`相关文档/相关小节` 是应当被检索出来的位置；')
    L.append('2. `类型=负样本`：语料里答不出来，**正确行为是拒答**，两列标注为`（应拒答）`；')
    L.append('3. `来源` 说明这条是谁造的，便于事后追溯「到底是谁考了谁」。\n')
    L.append('## 标注表\n')
    L.append('| # | 类型 | 问题 | 相关文档 | 相关小节 | 来源 |')
    L.append('|---|------|------|---------|---------|------|')
    n = 0
    for t in V1:
        n += 1
        L.append('| %d | 正样本 | %s | %s | %s | 人工 |' % (n, t['q'], t['doc'], t['section']))
    for t in SYNTH:
        n += 1
        L.append('| %d | 正样本 | %s | %s | %s | %s |' % (n, t['q'], t['doc'], t['section'], t['src']))
    for t in NEG:
        n += 1
        L.append('| %d | 负样本 | %s | %s | %s | %s |' % (n, t['q'], t['doc'], t['section'], t['src']))
    L.append('')
    L.append('## 统计\n')
    L.append('- 总计 %d 条：人工 %d + LLM 合成 %d + 对抗负例 %d' % (n, len(V1), len(SYNTH), len(NEG)))
    L.append('- 正样本平均字面命中率：人工 %.3f / 合成 %.3f'
             % (sum(t['lhr'] for t in V1) / len(V1),
                sum(t['lhr'] for t in SYNTH) / len(SYNTH) if SYNTH else 0.0))
    L.append('- 零成本基线 BM25@1 命中率：人工 %.3f / 合成 %.3f'
             % (sum(t['bm25_ok'] for t in V1) / len(V1),
                sum(t['bm25_ok'] for t in SYNTH) / len(SYNTH) if SYNTH else 0.0))
    L.append('- 负样本 top1 余弦均值：%.3f（正样本均值 %.3f）'
             % (sum(t['cos'] for t in NEG) / len(NEG) if NEG else 0.0,
                sum(t['cos'] for t in V1) / len(V1)))
    L.append('')
    L.append('## 已知局限\n')
    L.append('- 合成题只有 %d 条，样本量不足以单独下结论，只作方法示范；' % len(SYNTH))
    L.append('- 负样本是模型「设想」出来的，不是真实用户问过的，真实性弱于线上日志；')
    L.append('- 本文件不参与调参即无 holdout 划分，正式使用时应当切分。')
    return '\n'.join(L) + '\n'

if _HAS_KEY:
    _text = build_v2()
    V2_PATH.write_text(_text, encoding='utf-8')
    print('已写入 %s（%d 条）\n' % (V2_PATH, len(V1) + len(SYNTH) + len(NEG)))
    print('v1 vs v2 构成对比：')
    print(_col('', 20) + _col('v1', 8) + 'v2')
    print('-' * 34)
    for _label, _a, _b in [('总条数', len(V1), len(V1) + len(SYNTH) + len(NEG)),
                           ('人工标注', len(V1), len(V1)),
                           ('LLM 合成', 0, len(SYNTH)),
                           ('负样本/应拒答', 0, len(NEG)),
                           ('列数', 4, 6)]:
        print(_col(_label, 20) + _col(str(_a), 8) + str(_b))
    print('\n前 3 条负样本：')
    for t in NEG[:3]:
        print('   ', t['q'])
    # 反向验证：拿第 34 课那个 4 列正则回头解析 v2，把真实后果打出来
    _re34 = re.compile(r'^\|\s*(\d+)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|$')
    _hit = [m for m in (_re34.match(l) for l in _text.splitlines()) if m]
    _g = _hit[0].groups()
    print('\n反查：把 v2 喂回第 34 课那个 4 列正则，会怎样？')
    print('   能匹配上 %d 行 —— 表格看起来「解析成功」了，这才是最危险的地方。' % len(_hit))
    print('   但第 34 课的列映射是 (序号, 问题, 相关文档, 相关小节)，套到 6 列的 v2 上整体错位：')
    print('      问题   ← %r' % _g[1])
    print('      文档   ← %r' % _g[2])
    print('      小节   ← %r' % _g[3])
    print('   于是 `rel` 全为空，第 34 课那句 assert 会当场报错。')
    print('   好在是「跑不起来」，不是「悄悄算出一堆错数」——但无论如何，都得先改正则、')
    print('   打了版本号再切；本课到此为止不切换。')
else:
    recorded("""已写入 data\\评测集_v2.md（24 条）   ← 仅配了 Key 时才会真写，无 Key 不落盘

v1 vs v2 构成对比：
                    v1      v2
----------------------------------
总条数              15      24
人工标注            15      15
LLM 合成            0       4
负样本/应拒答       0       5
列数                4       6

前 3 条负样本：
    能和企业微信的审批流程打通吗？
    支持对接飞书多维表格做知识同步吗？
    有没有和容联七陌的客服系统对接案例？

反查：把 v2 喂回第 34 课那个 4 列正则，会怎样？
   能匹配上 24 行 —— 表格看起来「解析成功」了，这才是最危险的地方。
   但第 34 课的列映射是 (序号, 问题, 相关文档, 相关小节)，套到 6 列的 v2 上整体错位：
      问题   ← '正样本'
      文档   ← '星云客服机器人能私有化部署吗'
      小节   ← '星云智能产品手册.md | 部署方式 | 人工'
   于是 `rel` 全为空，第 34 课那句 assert 会当场报错。
   好在是「跑不起来」，不是「悄悄算出一堆错数」——但无论如何，都得先改正则、
   打了版本号再切；本课到此为止不切换。""",
             '录制于 2026-09-12，模型 qwen-plus')


已写入 data\评测集_v2.md（24 条）

v1 vs v2 构成对比：
                    v1      v2
----------------------------------
总条数              15      24
人工标注            15      15
LLM 合成            0       4
负样本/应拒答       0       5
列数                4       6

前 3 条负样本：
    能和企业微信的审批流程打通吗？
    支持对接飞书多维表格吗？
    跟阿里云百炼、腾讯混元比，推理速度差距大吗？

反查：把 v2 喂回第 34 课那个 4 列正则，会怎样？
   能匹配上 24 行 —— 表格看起来「解析成功」了，这才是最危险的地方。
   但第 34 课的列映射是 (序号, 问题, 相关文档, 相关小节)，套到 6 列的 v2 上整体错位：
      问题   ← '正样本'
      文档   ← '星云客服机器人能私有化部署吗'
      小节   ← '星云智能产品手册.md | 部署方式 | 人工'
   于是 `rel` 全为空，第 34 课那句 assert 会当场报错。
   好在是「跑不起来」，不是「悄悄算出一堆错数」——但无论如何，都得先改正则、
   打了版本号再切；本课到此为止不切换。


## 指标怎么读 & 小结

| 量 | 测什么 | 怎么用 |
|----|--------|--------|
| **字面命中率** | 问题有多像原文 | 均值偏高 → 这批题偏送分，考虑重出 |
| **BM25@1 命中率** | 零成本基线能拿几分 | 越高 → 这批题越测不出向量/重排的价值 |
| **负样本 top1 余弦** | 答不出来时系统有多「自信」 | 与正样本重叠 → 固定阈值分不开，得靠第 26/29 课那类分诊 |

三句话收尾：

1. **评测集本身就是交付物**——它决定你所有指标的可信度，值得单独一课；
2. **出题的人决定考题的价值**——自己出题自己考等于开卷，所以要来源标注、要 holdout；
3. **没有负样本的评测集是不完整的**——它永远测不出「该拒答时硬答」，而这恰恰是 RAG 上线后最贵的一类事故。

> 下一课（第 34 课）用 v1 算检索指标；第 35 课算生成指标；第 41 课做端到端总验收。
